In [19]:
import gradio as gr
import os
import json
import tempfile
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI


In [20]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")

client = OpenAI(
    api_key=openai_api_key,
    timeout=30.0,
    max_retries=2,
)

OpenAI API Key exists and begins sk-proj-


In [21]:
model = "gpt-5-nano"

def get_reply_from_llm(message):
    response = client.chat.completions.create(
        model=model,
        messages=message,
        reasoning_effort="minimal",
        response_format={"type": "json_object"},
        timeout=30.0,
    )
    return response.choices[0].message.content


def _parse_json_response(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        start = raw_text.find("{")
        end = raw_text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(raw_text[start : end + 1])
        raise


def _json_to_dataframe(payload):
    if isinstance(payload, dict) and "records" in payload and isinstance(payload["records"], list):
        return pd.DataFrame(payload["records"])
    if isinstance(payload, list):
        return pd.DataFrame(payload)
    if isinstance(payload, dict):
        return pd.json_normalize(payload)
    return pd.DataFrame([{"result": str(payload)}])


def _df_to_temp_csv(df: pd.DataFrame) -> str:
    import tempfile

    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False)
    df.to_csv(tmp.name, index=False)
    tmp.close()
    return tmp.name


def chat_and_generate_table(user_message, history):
    """Gradio callback.

    - Immediately appends the user's message to chat
    - Shows "Processing..." while the LLM runs
    - Clears the textbox right away
    """

    if history is None:
        history = []

    # Immediately reflect the user's message in the chat UI.
    history = history + [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": "Processing..."},
    ]

    # First yield: show processing + clear textbox.
    yield history, pd.DataFrame(), None, ""

    system_prompt = """You generate synthetic healthcare data.
Return ONLY valid JSON.

Always follow this schema (do not return any other top-level keys):
{
  "records": [
    {
      "patient_id": "P001",
      "modality": "CT",
      "sex": "Female",
      "age": 52,
      "BMI": 26.6,
      "height_cm": 168,
      "weight_kg": 75,
      "study_description": "Chest CT dose study",
      "protocol_used": "Standard",
      "CTDIvol_mGy": 6.4,
      "DLP_mGy_cm": 420,
      "effective_dose_mSv": 8.1,
      "rotation_time_s": 0.5,
      "pitch": 1.0,
      "tube_voltage_kVp": 120,
      "tube_current_mA": 240
    }
  ]
}

If the user asks for N rows, set records length to exactly N.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    try:
        raw = get_reply_from_llm(messages)
        payload = _parse_json_response(raw)
        df = _json_to_dataframe(payload)
        csv_path = _df_to_temp_csv(df)

        modality = None
        if isinstance(payload, dict) and isinstance(payload.get("records"), list) and payload["records"]:
            first = payload["records"][0]
            if isinstance(first, dict):
                modality = first.get("modality")
        if modality is None and isinstance(payload, dict):
            modality = payload.get("modality")

        assistant_text = f"Generated {len(df)} synthetic record(s)"
        if modality:
            assistant_text += f" for {modality}"
        assistant_text += ".\n\nTable is ready below. Use 'Download CSV' to export."

    except Exception as exc:
        df = pd.DataFrame([{"error": str(exc)}])
        csv_path = _df_to_temp_csv(df)
        assistant_text = (
            "I couldn't generate valid JSON this time. "
            "Please retry with a clear request like: 'Generate 10 CT records ...'."
            f" Error: {exc}"
        )

    # Replace the last "Processing..." message with the final reply.
    history[-1] = {"role": "assistant", "content": assistant_text}
    yield history, df, csv_path, ""


In [5]:
# Example manual call (optional):
# print(get_reply_from_llm(message))


{
  "modality": "CT",
  "study_description": "Synthetic CT dose-related dataset",
  "protocol_used": "Standard phantom-based protocol with varying tube current modulation",
  "patient_demographics": {
    "patient_id": "CT-DS-0001",
    "sex": "Female",
    "age_years": 52,
    "BMI": 26.8,
    "height_cm": 168,
    "weight_kg": 75
  },
  "study_metrics": {
    "study_date": "2026-03-27",
    "operator": "Synthetic Data Generator",
    "scanner_model": "CT-Synth 3000",
    "facility": "Synthetic Health Lab"
  },
  "dose_metrics": {
    "CTDIvol_mGy": 6.4,
    "DLP_mGy_cm": 420,
    "effective_dose_mSv": 8.1,
    "tube_voltage_kVp": 120,
    "tube_current_mA": 240,
    "rotation_time_s": 0.5,
    "pitch": 1.0,
    "collimation_mm": 64,
    "scan_length_mm": 370
  },
  "dose_variability": {
    "inter_scan_ratio": 1.05,
    "noise_index": 12,
    "adaptive_techniques": true,
    "dose_reduction_strategy": "Iterative reconstruction (IR) and tube current modulation"
  },
  "image_quality_m

In [22]:
with gr.Blocks(title="Synthetic Healthcare Data Chat") as demo:
    gr.Markdown("## Synthetic Healthcare Data Generator")
    chatbot = gr.Chatbot(type="messages", label="Conversation")
    user_input = gr.Textbox(
        label="Your request",
        placeholder="Example: Generate 10 CT records with BMI, height, weight, CTDIvol and DLP",
    )
    generate_btn = gr.Button("Generate Synthetic Data")
    output_table = gr.Dataframe(label="Generated Table", interactive=False)
    download_csv = gr.File(label="Download CSV")

    generate_btn.click(
        fn=chat_and_generate_table,
        inputs=[user_input, chatbot],
        outputs=[chatbot, output_table, download_csv, user_input],
    )
    user_input.submit(
        fn=chat_and_generate_table,
        inputs=[user_input, chatbot],
        outputs=[chatbot, output_table, download_csv, user_input],
    )

# Run this cell to launch UI
demo.launch()


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
